# Building the Frontend for a Full-Stack CrewAI App

## Step-by-Step Plan: From Backend API to a Web Page

In the previous notebook (`074-CREWAI-top5-backend.ipynb`), we built a **backend** — a FastAPI server with one endpoint:

```
POST /support/ticket  →  Receives a ticket, runs CrewAI, returns the AI response
```

But a backend alone is just a server waiting for requests. Users don't type `curl` commands — they use a **web page**.

**In this notebook, we will build the frontend** — a web page where users can:
1. Fill in a form (name, email, subject, message).
2. Click "Submit Ticket".
3. See the AI's response displayed on the page.

## What Is a Frontend?

A **frontend** is the part of the application that the user sees and interacts with in their browser. It's built with HTML, CSS, and JavaScript.

We'll use **Next.js** — the same React framework used in the todo app (`051-todoapp-LC-frontend`). Next.js makes it easy to build web pages with React components.

```
User's Browser (Frontend)              Server (Backend)
┌──────────────────────────┐           ┌──────────────────────┐
│                          │           │                      │
│  ┌────────────────────┐  │  POST     │  FastAPI             │
│  │ Ticket Form        │  │ ────────> │    └── CrewAI Flow   │
│  │  Name: ________    │  │  JSON     │         ├── Classify │
│  │  Email: _______    │  │           │         ├── Route    │
│  │  Subject: _____    │  │           │         └── Resolve  │
│  │  Message: _____    │  │  JSON     │                      │
│  │  [Submit Ticket]   │  │ <──────── │  Returns response    │
│  └────────────────────┘  │           │                      │
│                          │           └──────────────────────┘
│  ┌────────────────────┐  │
│  │ AI Response Box    │  │  ← Shows the AI's response
│  │  Category: ...     │  │
│  │  Urgency: ...      │  │
│  │  Message: ...      │  │
│  └────────────────────┘  │
│                          │
└──────────────────────────┘
```

The frontend runs on `http://localhost:3000` and calls the backend at `http://localhost:8000`.

## The Big Picture: What We're Building

Our frontend is simple — just **one page** with:
- A **form** to submit a support ticket.
- A **response area** that shows the AI's answer after submission.

### Project Structure

Following the same pattern as the todo app frontend, our folder will look like this:

```
074-crewai-frontend/
├── .env                      ← Backend URL (http://localhost:8000)
├── package.json              ← Project dependencies (Next.js, React)
├── next.config.mjs           ← Next.js configuration
├── jsconfig.json             ← Path aliases
├── pages/
│   ├── _app.js               ← App wrapper (loads global CSS)
│   ├── _document.js          ← HTML document structure
│   └── index.js              ← Main page (our ticket form)
├── components/
│   ├── layout.js             ← Page layout (title + container)
│   ├── ticket-form.js        ← The form component
│   └── ticket-response.js    ← The response display component
└── styles/
    ├── globals.css            ← Global styles
    ├── layout.module.css      ← Layout styles
    ├── ticket-form.module.css ← Form styles
    └── ticket-response.module.css ← Response styles
```

**Comparison with the todo app:**

| Todo App | Our App | Purpose |
|----------|--------|---------|
| `todo-list.js` | `ticket-form.js` | Main interactive component |
| `todo.js` | `ticket-response.js` | Display component for results |
| `layout.js` | `layout.js` | Page layout wrapper |
| `index.js` | `index.js` | Main page |

Same structure, different components.

## Step 0: Create the Next.js Project

First, we need to create a new Next.js project. Open a terminal and run:

```bash
npx create-next-app@latest 074-crewai-frontend
```

When prompted, choose these options:
- TypeScript: **No**
- ESLint: **Yes**
- Tailwind CSS: **No**
- `src/` directory: **No**
- App Router: **No** (we use the Pages Router, same as the todo app)
- Import alias: **Yes** (keep the default `@/*`)

This creates a new folder `074-crewai-frontend/` with all the Next.js boilerplate.

Then navigate into the folder:

```bash
cd 074-crewai-frontend
```

## Step 1: The `.env` File

Just like the todo app, we need a `.env` file to tell the frontend where the backend is running.

In Next.js, environment variables that start with `NEXT_PUBLIC_` are available in the browser code.

In [ ]:
%%writefile 074-crewai-frontend/.env
NEXT_PUBLIC_API_URL=http://localhost:8000

**This is identical to the todo app's `.env` file.** The frontend will use `process.env.NEXT_PUBLIC_API_URL` to build the URL for API calls.

For example: `process.env.NEXT_PUBLIC_API_URL + '/support/ticket'` becomes `http://localhost:8000/support/ticket`.

## Step 2: `next.config.mjs` — Next.js Configuration

This file is the same as the todo app — just basic configuration.

In [ ]:
%%writefile 074-crewai-frontend/next.config.mjs
/** @type {import('next').NextConfig} */
const nextConfig = {
  reactStrictMode: true,
};

export default nextConfig;

## Step 3: `jsconfig.json` — Path Aliases

This lets us use `@/` as a shortcut for the project root. For example, `@/styles/globals.css` instead of `../../styles/globals.css`. Same as the todo app.

In [ ]:
%%writefile 074-crewai-frontend/jsconfig.json
{
  "compilerOptions": {
    "paths": {
      "@/*": ["./*"]
    }
  }
}

## Step 4: `pages/_app.js` — App Wrapper

This is the root component that wraps every page. Its main job is to load the global CSS. Identical to the todo app.

In [ ]:
%%writefile 074-crewai-frontend/pages/_app.js
import "@/styles/globals.css";

export default function App({ Component, pageProps }) {
  return <Component {...pageProps} />;
}

## Step 5: `pages/_document.js` — HTML Document Structure

This defines the base HTML structure. Identical to the todo app.

In [ ]:
%%writefile 074-crewai-frontend/pages/_document.js
import { Html, Head, Main, NextScript } from "next/document";

export default function Document() {
  return (
    <Html lang="en">
      <Head />
      <body>
        <Main />
        <NextScript />
      </body>
    </Html>
  );
}

## Step 6: `styles/globals.css` — Global Styles

Basic global styles for the entire app. We keep it simple — just a clean font and some reset styles.

In [ ]:
%%writefile 074-crewai-frontend/styles/globals.css
* {
  box-sizing: border-box;
  margin: 0;
  padding: 0;
}

body {
  font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;
  background-color: #f5f5f5;
  color: #333;
}

## Step 7: `styles/layout.module.css` — Layout Styles

Styles for the page layout (centered container with a title). Similar to the todo app's layout styles, but a bit wider since we have a form.

In [ ]:
%%writefile 074-crewai-frontend/styles/layout.module.css
.layout {
    max-width: 700px;
    margin: 40px auto;
    padding: 0 20px;
}

.title {
    text-align: center;
    font-size: 28px;
    margin-bottom: 30px;
    color: #1a1a1a;
}

**Comparison with the todo app's `layout.module.css`:**

| Todo App | Our App |
|----------|--------|
| `width: 500px` | `max-width: 700px` (wider for the form) |
| `margin: 20px` | `margin: 40px auto` (centered on page) |

Same idea, slightly adjusted sizing.

## Step 8: `styles/ticket-form.module.css` — Form Styles

Styles for the support ticket form. This is similar to how the todo app styled its input field and buttons, but adapted for a full form.

In [ ]:
%%writefile 074-crewai-frontend/styles/ticket-form.module.css
.form {
    background: white;
    border: 1px solid #ddd;
    border-radius: 8px;
    padding: 30px;
    margin-bottom: 20px;
}

.fieldGroup {
    margin-bottom: 15px;
}

.label {
    display: block;
    font-weight: 600;
    margin-bottom: 5px;
    font-size: 14px;
}

.input {
    width: 100%;
    padding: 10px;
    border: 1px solid #ccc;
    border-radius: 4px;
    font-size: 14px;
}

.textarea {
    width: 100%;
    padding: 10px;
    border: 1px solid #ccc;
    border-radius: 4px;
    font-size: 14px;
    min-height: 100px;
    resize: vertical;
}

.submitBtn {
    width: 100%;
    padding: 12px;
    background-color: #4a90d9;
    color: white;
    border: none;
    border-radius: 4px;
    font-size: 16px;
    cursor: pointer;
    margin-top: 10px;
}

.submitBtn:hover {
    background-color: #357abd;
}

.submitBtn:disabled {
    background-color: #aaa;
    cursor: not-allowed;
}

.loading {
    text-align: center;
    padding: 20px;
    color: #666;
    font-style: italic;
}

## Step 9: `styles/ticket-response.module.css` — Response Display Styles

Styles for displaying the AI's response. This is similar to the todo app's poem popup, but displayed inline on the page instead of as a modal.

In [ ]:
%%writefile 074-crewai-frontend/styles/ticket-response.module.css
.responseBox {
    background: white;
    border: 1px solid #ddd;
    border-radius: 8px;
    padding: 30px;
    margin-top: 20px;
}

.responseTitle {
    font-size: 20px;
    margin-bottom: 15px;
    color: #1a1a1a;
}

.metaRow {
    display: flex;
    gap: 15px;
    margin-bottom: 15px;
    flex-wrap: wrap;
}

.badge {
    padding: 4px 12px;
    border-radius: 12px;
    font-size: 13px;
    font-weight: 600;
}

.badgeSimple {
    background-color: #d4edda;
    color: #155724;
}

.badgeComplex {
    background-color: #fff3cd;
    color: #856404;
}

.badgeLow {
    background-color: #d4edda;
    color: #155724;
}

.badgeMedium {
    background-color: #fff3cd;
    color: #856404;
}

.badgeHigh {
    background-color: #f8d7da;
    color: #721c24;
}

.badgeCategory {
    background-color: #d1ecf1;
    color: #0c5460;
}

.subject {
    font-size: 16px;
    font-weight: 600;
    margin-bottom: 10px;
}

.message {
    line-height: 1.6;
    white-space: pre-wrap;
    background-color: #f9f9f9;
    padding: 15px;
    border-radius: 4px;
    border-left: 3px solid #4a90d9;
}

.followUp {
    margin-top: 15px;
    padding: 10px;
    border-radius: 4px;
    font-size: 14px;
    font-weight: 600;
}

.followUpYes {
    background-color: #fff3cd;
    color: #856404;
}

.followUpNo {
    background-color: #d4edda;
    color: #155724;
}

## Step 10: `components/layout.js` — Page Layout Component

This is the layout wrapper that provides a title and centers the content on the page. It's the same pattern as the todo app's `layout.js`.

In [ ]:
%%writefile 074-crewai-frontend/components/layout.js
import styles from '../styles/layout.module.css'

export default function Layout(props) {
  return (
    <div className={styles.layout}>
      <h1 className={styles.title}>BigCompany Inc. Support</h1>
      {props.children}
    </div>
  )
}

**Comparison with the todo app's `layout.js`:**

| Todo App | Our App |
|----------|--------|
| `<h1>To Do</h1>` | `<h1>BigCompany Inc. Support</h1>` |

Same component, different title. That's it.

## Step 11: `components/ticket-response.js` — Response Display Component

This component receives the AI's response data and displays it nicely. It shows:
- Colored badges for category, complexity, and urgency.
- The response subject and message.
- Whether a human follow-up is needed.

In the todo app, the `todo.js` component displayed a single todo item. Here, `ticket-response.js` displays the AI's response.

In [ ]:
%%writefile 074-crewai-frontend/components/ticket-response.js
import styles from '../styles/ticket-response.module.css'

export default function TicketResponse({ response }) {
  // Pick the right CSS class based on the value
  const complexityClass = response.complexity === 'simple'
    ? styles.badgeSimple
    : styles.badgeComplex;

  const urgencyClass = response.urgency === 'low'
    ? styles.badgeLow
    : response.urgency === 'medium'
      ? styles.badgeMedium
      : styles.badgeHigh;

  return (
    <div className={styles.responseBox}>
      <h2 className={styles.responseTitle}>AI Response</h2>

      {/* Colored badges for category, complexity, urgency */}
      <div className={styles.metaRow}>
        <span className={`${styles.badge} ${styles.badgeCategory}`}>
          {response.category}
        </span>
        <span className={`${styles.badge} ${complexityClass}`}>
          {response.complexity} path
        </span>
        <span className={`${styles.badge} ${urgencyClass}`}>
          {response.urgency} urgency
        </span>
      </div>

      {/* Response subject and message */}
      <div className={styles.subject}>{response.resolution_subject}</div>
      <div className={styles.message}>{response.resolution_message}</div>

      {/* Follow-up indicator */}
      <div className={`${styles.followUp} ${
        response.follow_up_needed ? styles.followUpYes : styles.followUpNo
      }`}>
        {response.follow_up_needed
          ? 'Human follow-up needed'
          : 'Fully resolved by AI — no follow-up needed'}
      </div>
    </div>
  )
}

**How this compares to the todo app's `todo.js`:**

| Todo App (`todo.js`) | Our App (`ticket-response.js`) |
|---------------------|-------------------------------|
| Receives a `todo` prop | Receives a `response` prop |
| Shows checkbox + text input | Shows badges + message |
| Has "Generate Poem" button (calls backend) | No extra actions — just displays data |
| Uses portal for poem popup | No popup — inline display |

Our component is simpler: it just receives data and displays it. No user interaction needed.

## Step 12: `components/ticket-form.js` — The Form Component

This is the main interactive component — the heart of our frontend. It handles:
1. **Form state** — tracking what the user types in each field.
2. **Submitting the ticket** — sending the data to the backend.
3. **Loading state** — showing "Processing..." while the AI works.
4. **Displaying the response** — passing the result to the `TicketResponse` component.

In the todo app, `todo-list.js` was the main component. It handled:
- Adding todos (POST to backend)
- Fetching todos (GET from backend)
- Updating todos (PUT to backend)
- Deleting todos (DELETE to backend)

Our component is simpler — it only does one thing: **POST a ticket and display the response**.

In [ ]:
%%writefile 074-crewai-frontend/components/ticket-form.js
import styles from '../styles/ticket-form.module.css'
import { useState } from 'react'
import TicketResponse from './ticket-response'

export default function TicketForm() {
  // Form fields — same idea as the todo app's mainInput state
  const [name, setName] = useState('')
  const [email, setEmail] = useState('')
  const [subject, setSubject] = useState('')
  const [message, setMessage] = useState('')

  // Loading and response state
  const [loading, setLoading] = useState(false)
  const [response, setResponse] = useState(null)

  // Submit the ticket to the backend
  async function handleSubmit(e) {
    e.preventDefault()  // Prevent the page from refreshing

    setLoading(true)
    setResponse(null)

    // POST to the backend — same pattern as the todo app's addToDo function
    const res = await fetch(process.env.NEXT_PUBLIC_API_URL + '/support/ticket', {
      method: 'POST',
      body: JSON.stringify({
        customer_name: name,
        customer_email: email,
        ticket_subject: subject,
        ticket_message: message,
      }),
      headers: {
        'Content-Type': 'application/json'
      }
    })

    if (res.ok) {
      const data = await res.json()
      setResponse(data)
    }

    setLoading(false)
  }

  return (
    <div>
      {/* The ticket form */}
      <form className={styles.form} onSubmit={handleSubmit}>
        <div className={styles.fieldGroup}>
          <label className={styles.label}>Your Name</label>
          <input
            className={styles.input}
            type="text"
            value={name}
            onChange={(e) => setName(e.target.value)}
            placeholder="e.g. Alice Johnson"
            required
          />
        </div>

        <div className={styles.fieldGroup}>
          <label className={styles.label}>Your Email</label>
          <input
            className={styles.input}
            type="email"
            value={email}
            onChange={(e) => setEmail(e.target.value)}
            placeholder="e.g. alice@example.com"
            required
          />
        </div>

        <div className={styles.fieldGroup}>
          <label className={styles.label}>Subject</label>
          <input
            className={styles.input}
            type="text"
            value={subject}
            onChange={(e) => setSubject(e.target.value)}
            placeholder="e.g. Can't reset my password"
            required
          />
        </div>

        <div className={styles.fieldGroup}>
          <label className={styles.label}>Message</label>
          <textarea
            className={styles.textarea}
            value={message}
            onChange={(e) => setMessage(e.target.value)}
            placeholder="Describe your issue in detail..."
            required
          />
        </div>

        <button
          className={styles.submitBtn}
          type="submit"
          disabled={loading}
        >
          {loading ? 'Processing...' : 'Submit Ticket'}
        </button>
      </form>

      {/* Loading message */}
      {loading && (
        <div className={styles.loading}>
          The AI crew is working on your ticket. This may take a minute...
        </div>
      )}

      {/* Display the AI response */}
      {response && <TicketResponse response={response} />}
    </div>
  )
}

### How the `fetch` Call Compares to the Todo App

The core pattern is identical. Let's compare the `fetch` call in both apps:

**Todo app** (`todo-list.js` — `addToDo` function):
```javascript
const res = await fetch(process.env.NEXT_PUBLIC_API_URL + '/todos/', {
  method: 'POST',
  body: JSON.stringify({
    name: name,
    completed: false
  }),
  headers: {
    'Content-Type': 'application/json'
  }
})
```

**Our app** (`ticket-form.js` — `handleSubmit` function):
```javascript
const res = await fetch(process.env.NEXT_PUBLIC_API_URL + '/support/ticket', {
  method: 'POST',
  body: JSON.stringify({
    customer_name: name,
    customer_email: email,
    ticket_subject: subject,
    ticket_message: message,
  }),
  headers: {
    'Content-Type': 'application/json'
  }
})
```

Same structure:
1. `fetch()` with the backend URL + endpoint path.
2. `method: 'POST'` to send data.
3. `body: JSON.stringify(...)` to convert our data to JSON.
4. `headers` to tell the backend we're sending JSON.

The only difference is the URL (`/todos/` vs `/support/ticket`) and the data fields.

## Step 13: `pages/index.js` — The Main Page

This is the main page of our app. It brings everything together:
- Uses the `Layout` component for the page structure.
- Uses the `TicketForm` component for the form and response.

This follows the exact same pattern as the todo app's `index.js`.

In [ ]:
%%writefile 074-crewai-frontend/pages/index.js
import Head from 'next/head'
import Layout from '../components/layout'
import TicketForm from '../components/ticket-form'

export default function Home() {
  return (
    <div>
      <Head>
        <title>BigCompany Inc. Support</title>
        <meta name="description" content="AI-Powered Customer Support" />
        <link rel="icon" href="/favicon.ico" />
      </Head>
      <Layout>
        <TicketForm />
      </Layout>
    </div>
  )
}

**Comparison with the todo app's `index.js`:**

```javascript
// Todo app
import ToDoList from '../components/todo-list'
<Layout>
  <ToDoList />
</Layout>

// Our app
import TicketForm from '../components/ticket-form'
<Layout>
  <TicketForm />
</Layout>
```

Identical structure. We just swapped `ToDoList` for `TicketForm`.

## Step 14: How to Run the Frontend

**Prerequisites:** Make sure you have Node.js installed and the backend is running (`uvicorn main:app --reload` in the `074-crewai-backend/` folder).

Then, in a **new terminal**, navigate to the `074-crewai-frontend/` folder and run:

```bash
npm install
npm run dev
```

This starts the frontend at `http://localhost:3000`.

You should see:

```
  ▲ Next.js
  - Local:   http://localhost:3000
```

Open `http://localhost:3000` in your browser and you'll see the support ticket form.

## Step 15: Running the Full Stack App

To run the complete application, you need **two terminals** running at the same time:

### Terminal 1 — Backend (port 8000)
```bash
cd 074-crewai-backend
uvicorn main:app --reload
```

### Terminal 2 — Frontend (port 3000)
```bash
cd 074-crewai-frontend
npm run dev
```

Then open `http://localhost:3000` in your browser.

```
┌─────────────────────────────────────────────────────┐
│                                                     │
│  Terminal 1 (Backend)      Terminal 2 (Frontend)    │
│  ┌──────────────────┐      ┌──────────────────┐     │
│  │ uvicorn main:app │      │ npm run dev      │     │
│  │ --reload         │      │                  │     │
│  │                  │      │                  │     │
│  │ Running on       │      │ Running on       │     │
│  │ localhost:8000   │ ◄──► │ localhost:3000   │     │
│  └──────────────────┘      └──────────────────┘     │
│                                                     │
│  Browser: http://localhost:3000                     │
│                                                     │
└─────────────────────────────────────────────────────┘
```

### How to test it:

1. Fill in the form:
   - Name: `Alice Johnson`
   - Email: `alice@example.com`
   - Subject: `Can't log in — need password reset`
   - Message: `Hi, I forgot my password and can't log into my account. How do I reset it?`

2. Click **Submit Ticket**.

3. Wait for the AI to process (this takes a minute — the CrewAI Flow is running on the backend).

4. See the AI's response appear below the form with:
   - Category badge (e.g., `password_reset`)
   - Complexity badge (e.g., `simple path`)
   - Urgency badge (e.g., `low urgency`)
   - The full response message
   - Whether a human follow-up is needed

## Recap: The Complete Full-Stack Architecture

Here's how everything fits together:

```
┌──────────────────────────────────────────────────────────────────────┐
│  FRONTEND (Next.js — localhost:3000)                                 │
│                                                                      │
│  pages/index.js                                                      │
│    └── components/layout.js          ← Page wrapper (title)          │
│    └── components/ticket-form.js     ← Form + fetch() to backend     │
│          └── components/ticket-response.js  ← Displays AI response   │
│                                                                      │
│  .env → NEXT_PUBLIC_API_URL=http://localhost:8000                    │
└──────────────────────────────┬───────────────────────────────────────┘
                               │
                    POST /support/ticket
                         (JSON)
                               │
                               ▼
┌──────────────────────────────────────────────────────────────────────┐
│  BACKEND (FastAPI — localhost:8000)                                  │
│                                                                      │
│  main.py                             ← FastAPI app + CORS            │
│    └── routers/support.py            ← POST /support/ticket endpoint │
│          └── crew_logic.py           ← CrewAI Flow + Agents          │
│                ├── Classifier Agent   ← Classifies the ticket        │
│                ├── FAQ Agent          ← Answers simple questions     │
│                ├── Research Agent     ← Investigates complex issues  │
│                └── Solution Agent     ← Writes empathetic responses  │
│                                                                      │
│  .env → OPENAI_API_KEY=sk-...                                        │
└──────────────────────────────────────────────────────────────────────┘
```

### File-by-File Summary

| File | What It Does | Inspired By |
|------|-------------|------------|
| `.env` | Backend URL | `051-todoapp-LC-frontend/.env` |
| `next.config.mjs` | Next.js config | `051-todoapp-LC-frontend/next.config.mjs` |
| `jsconfig.json` | Path aliases | `051-todoapp-LC-frontend/jsconfig.json` |
| `pages/_app.js` | App wrapper, loads CSS | `051-todoapp-LC-frontend/pages/_app.js` |
| `pages/_document.js` | HTML structure | `051-todoapp-LC-frontend/pages/_document.js` |
| `pages/index.js` | Main page | `051-todoapp-LC-frontend/pages/index.js` |
| `components/layout.js` | Layout wrapper | `051-todoapp-LC-frontend/components/layout.js` |
| `components/ticket-form.js` | Form + API call | `051-todoapp-LC-frontend/components/todo-list.js` |
| `components/ticket-response.js` | Response display | `051-todoapp-LC-frontend/components/todo.js` |
| `styles/*.module.css` | Component styles | `051-todoapp-LC-frontend/styles/*.module.css` |

## Key Takeaways for Beginners

1. **The frontend and backend are separate programs.** The frontend runs on port 3000 (Next.js), the backend runs on port 8000 (FastAPI). They talk to each other via HTTP requests (JSON).

2. **`fetch()` is how the frontend talks to the backend.** It sends a POST request with the ticket data and receives the AI's response. This is the same `fetch()` pattern used in the todo app.

3. **CORS makes this possible.** Without CORS configuration on the backend, the browser would block requests from `localhost:3000` to `localhost:8000`. That's why `main.py` includes `CORSMiddleware`.

4. **`.env` files store configuration.** The frontend uses `NEXT_PUBLIC_API_URL` to know where the backend is. The backend uses `OPENAI_API_KEY` to authenticate with OpenAI.

5. **React state (`useState`) tracks what the user types.** Each form field has its own state variable. When the user types, the state updates. When they submit, the state values are sent to the backend.

6. **Components are reusable building blocks.** `Layout` wraps the page, `TicketForm` handles the form, `TicketResponse` displays the result. Each component has its own CSS module.

7. **The project structure follows the same pattern as the todo app.** If you understood the todo app, you already understand 90% of this app. The only new thing is the form and the response display.